<a href="https://colab.research.google.com/github/shubhamgarg190110/SP_500_Research_Project/blob/main/sp500_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

## 1. Read in the dataset

In [ ]:
url = "https://raw.githubusercontent.com/shubhamgarg190110/dsrpsp500/refs/heads/main/sp500.csv"
local_file = Path("sp500.csv")

if local_file.exists():
    raw_df = pd.read_csv(local_file)
else:
    raw_df = pd.read_csv(url)

print(f"Raw shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
raw_df.head()


Raw shape: 1,865 rows × 10 columns


,Date,SP500,Dividend,Earnings,Consumer Price Index,Long Interest Rate,Real Price,Real Dividend,Real Earnings,PE10
0,1871-01-01,4.44,0.26,0.4,12.46,5.32,109.05,6.39,9.82,0.0
1,1871-02-01,4.50,0.26,0.4,12.84,5.32,107.25,6.20,9.53,0.0
2,1871-03-01,4.61,0.26,0.4,13.03,5.33,108.27,6.11,9.39,0.0
3,1871-04-01,4.74,0.26,0.4,12.56,5.33,115.54,6.34,9.75,0.0
4,1871-05-01,4.86,0.26,0.4,12.27,5.33,121.22,6.48,9.98,0.0


## 2. Inspect the raw data

In [ ]:
raw_df.info()

raw_quality = pd.DataFrame({
    "data_type": raw_df.dtypes.astype(str),
    "missing_count": raw_df.isna().sum(),
    "missing_percent": (raw_df.isna().mean() * 100).round(2),
    "unique_values": raw_df.nunique(dropna=False)
})
raw_quality


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1865 entries, 0 to 1864
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Date                  1865 non-null   object 
 1   SP500                 1865 non-null   float64
 2   Dividend              1865 non-null   float64
 3   Earnings              1865 non-null   float64
 4   Consumer Price Index  1865 non-null   float64
 5   Long Interest Rate    1865 non-null   float64
 6   Real Price            1865 non-null   float64
 7   Real Dividend         1865 non-null   float64
 8   Real Earnings         1865 non-null   float64
 9   PE10                  1865 non-null   float64
dtypes: float64(9), object(1)
memory usage: 145.8+ KB


,data_type,missing_count,missing_percent,unique_values
Date,object,0,0.0,1865
SP500,float64,0,0.0,1494
Dividend,float64,0,0.0,1191
Earnings,float64,0,0.0,1389
Consumer Price Index,float64,0,0.0,889
Long Interest Rate,float64,0,0.0,649
Real Price,float64,0,0.0,1783
Real Dividend,float64,0,0.0,1254
Real Earnings,float64,0,0.0,1548
PE10,float64,0,0.0,1204


## 3. Clean column names and data types

In [18]:
df = raw_df.copy()

def clean_column_name(name):
    """Convert a column name to lowercase snake_case."""
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

df.columns = [clean_column_name(col) for col in df.columns]

# Parse the monthly date variable.
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Coerce all remaining variables to numeric.
numeric_columns = df.columns.drop("date")
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

df.dtypes


,0
date,datetime64[ns]
sp500,float64
dividend,float64
earnings,float64
consumer_price_index,float64
long_interest_rate,float64
real_price,float64
real_dividend,float64
real_earnings,float64
pe10,float64


## 4. Handle duplicates and missing data

`PE10` is a valuation ratio and a value of zero is not meaningful here. The source uses zero as a placeholder during periods when PE10 is unavailable, so those zeros are converted to `NaN`.

The missing PE10 observations are retained as missing rather than interpolated because estimating a valuation ratio would introduce artificial data.

### Row removal

Rows 1–121 and rows 1832–1866 are removed using 1-based row numbering due to these rows containg missing data. Because the dataset contains 1,865 rows, the second range effectively removes rows 1832–1865.

In [19]:
# Remove specified rows using 1-based row numbering.
# Rows 1-121 and 1832-1866 are removed.
# If the dataset has fewer than 1866 rows, removal stops at the final row.

rows_before = len(df)

positions_to_remove = set(range(0, min(121, rows_before)))
positions_to_remove.update(range(1831, min(1866, rows_before)))

row_mask = ~pd.Series(range(rows_before)).isin(positions_to_remove)
df = df.loc[row_mask.values].reset_index(drop=True)

print(f"Rows before removal: {rows_before}")
print(f"Rows removed: {rows_before - len(df)}")
print(f"Rows remaining: {len(df)}")

Rows before removal: 1865
Rows removed: 155
Rows remaining: 1710


## 5. Confirm tidy structure and data quality


In [20]:
clean_quality = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=False)
})

clean_quality

,data_type,missing_count,missing_percent,unique_values
date,datetime64[ns],0,0.0,1710
sp500,float64,0,0.0,1414
dividend,float64,0,0.0,1172
earnings,float64,0,0.0,1358
consumer_price_index,float64,0,0.0,861
long_interest_rate,float64,0,0.0,627
real_price,float64,0,0.0,1669
real_dividend,float64,0,0.0,1180
real_earnings,float64,0,0.0,1482
pe10,float64,0,0.0,1201


In [21]:
#%%
# Tidy-data checks:
# 1 row = 1 monthly observation
# 1 column = 1 variable
assert df.columns.is_unique, "Column names are not unique."
assert df["date"].is_unique, "There is more than one row for a month."
assert df["date"].is_monotonic_increasing, "Dates are not sorted."
assert not df.drop(columns="pe10").isna().any().any(), (
    "Unexpected missing values remain outside PE10."
)

expected_columns = [
    "date",
    "sp500",
    "dividend",
    "earnings",
    "consumer_price_index",
    "long_interest_rate",
    "real_price",
    "real_dividend",
    "real_earnings",
    "pe10",
]
assert df.columns.tolist() == expected_columns, "Unexpected columns or column order."

print(f"Clean shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print("The dataset is tidy: each row is one month and each column is one variable.")

df.head()



Clean shape: 1,710 rows × 10 columns
Date range: 1881-02-01 to 2023-07-01
The dataset is tidy: each row is one month and each column is one variable.


,date,sp500,dividend,earnings,consumer_price_index,long_interest_rate,real_price,real_dividend,real_earnings,pe10
0,1881-02-01,6.17,0.270,0.4817,9.51,3.69,198.52,8.69,15.50,18.15
1,1881-03-01,6.24,0.275,0.4775,9.51,3.69,200.77,8.85,15.36,18.27
2,1881-04-01,6.22,0.280,0.4733,9.61,3.68,198.15,8.92,15.08,17.95
3,1881-05-01,6.50,0.285,0.4692,9.51,3.67,209.13,9.17,15.10,18.87
4,1881-06-01,6.58,0.290,0.4650,9.51,3.67,211.71,9.33,14.96,19.03


## 6. Save the cleaned dataset


In [23]:
output_file = Path("sp500_cleaned.csv")
df.to_csv(output_file, index=False, date_format="%Y-%m-%d")

print(f"Saved cleaned data to: {output_file.resolve()}")


Saved cleaned data to: /content/sp500_cleaned.csv
